<h1 style="text-align: center;">Baseline Model</h1>
<h3 style="text-align: center;">Hotel Booking Cancellation Prediction</h3>

---

<h5 style="text-align: right;">By Beta Group</h5>

## Pemilihan Model

Berdasarkan hasil ROC-AUC, **Logistic Regression (base) dan XGBoost** dipilih sebagai dua model untuk eksperimen lebih lanjut.

* **XGBoost** memperoleh ROC-AUC validasi tertinggi, yaitu (**0,8276**). Namun, terdapat selisih yang cukup besar antara skor training (**0,9954**) dan skor validation, yang mengindikasikan adanya overfitting.

* **Logistic Regression (base)** memperoleh ROC-AUC validasi tertinggi kedua, yaitu (**0,8223**). Model ini menunjukkan indikasi overfitting yang lebih rendah dibandingkan XGBoost, meskipun masih terdapat selisih antara ROC-AUC training (**0,9742**) dan validation.

Oleh karena itu, **XGBoost dan Logistic Regression (base)** akan dilanjutkan ke tahap hyperparameter tuning dan evaluasi lebih lanjut.

In [1]:
import warnings
warnings.filterwarnings("ignore")

# **Section 0. Setup**
## **0.1 Import Library**

In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import PredefinedSplit
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

## **0.2 Global Configuration**

In [3]:
RANDOM_STATE=42
READ_CSV = "../data/hotel_booking_2017_cleaned.csv"

In [4]:
NUMERIC_COLS = [
    "lead_time",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
]

CATEGORICAL_COLS = [
    "hotel",
    "arrival_date_year",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
    "agent",
    "company",

    "has_children",
    "has_babies",
    "is_family",
    "has_agent",
    "has_company",
    "room_type_changed"
]

# **Section 1. Load Dataset**

In [5]:
df = pd.read_csv(READ_CSV)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,has_babies,is_family,total_stay_nights,has_agent,has_company,total_previous_bookings,previous_cancellation_rate,has_previous_cancellation,has_booking_changes,room_type_changed
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,0,0,0,0,0,0.0,0,1,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,0,0,0,0,0,0.0,0,1,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,0,1,0,0,0,0.0,0,0,1
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,0,1,1,0,0,0.0,0,0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,0,2,1,0,0,0.0,0,0,0


# **Section 2. Data Splitting**

In [6]:
train_df = df[df["arrival_date_year"] == 2015].copy()
val_df   = df[df["arrival_date_year"] == 2016].copy()
test_df  = df[df["arrival_date_year"] == 2017].copy()

x_train = train_df.drop(columns=["is_canceled"])
y_train = train_df["is_canceled"]

x_val = val_df.drop(columns=["is_canceled"])
y_val = val_df["is_canceled"]

x_test = test_df.drop(columns=["is_canceled"])
y_test = test_df["is_canceled"]

print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

Train      : 21,965
Validation : 56,621
Test       : 40,619


In [7]:
# Save splitted datasets to CSV
os.makedirs('../data/split', exist_ok=True)
train_df.to_csv('../data/split/train.csv')
val_df.to_csv('../data/split/validation.csv')
test_df.to_csv('../data/split/test.csv')

In [8]:
search_df = pd.concat(
    [train_df, val_df],
    axis=0
).reset_index(drop=True)

x_search = search_df.drop(columns=["is_canceled"])
y_search = search_df["is_canceled"]

# -1 = selalu masuk training
#  0 = validation fold
test_fold = np.where(
    search_df["arrival_date_year"] == 2016,
    0,
    -1
)

temporal_cv = PredefinedSplit(
    test_fold=test_fold
)

print("CV split:")
print(f"Training samples   : {(test_fold == -1).sum():,}")
print(f"Validation samples : {(test_fold == 0).sum():,}")

CV split:
Training samples   : 21,965
Validation samples : 56,621


# **Section 3. Feature Engineering Pipeline**

In [9]:
numeric_pipeline = Pipeline([
    ("RobustScaler", RobustScaler()),
])

Di tahap *data cleaning*, outlier pada `lead_time`, `adr`, `previous_cancellations`, `booking_changes`, `required_car_parking_spaces`, dan lain-lain sengaja dipertahankan karena masih masuk akal secara bisnis. Karena outlier ini tetap ada, scaler yang dipakai perlu tahan terhadap outlier tersebut, dan `RobustScaler` dipakai karena tidak terpengaruh terhadap nilai ekstrem tunggal.

- `StandardScaler` pakai mean & std → outlier ekstrem menggeser mean dan menekan variasi titik-titik normal.
- `MinMaxScaler` pakai min & max → satu titik ekstrem langsung menentukan skala seluruh kolom.
- `RobustScaler` pakai median & IQR → tidak terpengaruh nilai ekstrem tunggal.

In [10]:
categorical_ohe_pipeline = Pipeline([
    ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

Split yang dipakai di bersifat **temporal** (train = 2015, validation = 2016, test = 2017), bukan random split. Konsekuensinya, kategori pada `agent`, `company`, atau `country` di tahun 2016/2017 tidak dijamin identik dengan kategori yang terlihat model saat `.fit()` di data 2015. Tanpa `handle_unknown="ignore"`, `OneHotEncoder` akan **error** begitu menemukan kategori yang belum pernah dilihat. Dengan `ignore`, baris berkategori baru tetap bisa diproses, sehingga pipeline tidak *crash* saat dievaluasi ke val/test.

Baseline model empat varian Logistic Regression (base, Lasso, Ridge, ElasticNet). Kalau seluruh kategori sebuah fitur di-one-hot tanpa drop satu, kolom-kolom dummy itu akan selalu berjumlah 1, membuat matriks desain redundan/singular untuk model linear. `drop="first"` membuang satu kategori sebagai baseline referensi untuk menghindari redundansi ini.

In [11]:
preprocessor = ColumnTransformer(
    [
        ("numeric_pipeline", numeric_pipeline, NUMERIC_COLS),
        ("categorical_ohe_pipeline", categorical_ohe_pipeline, CATEGORICAL_COLS),
    ],
    remainder="drop"
)

In [12]:
pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression())
    ]
)

# **Section 4. Modeling**

In [13]:
logreg_base = LogisticRegression(random_state=RANDOM_STATE)
logreg_lasso = LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
logreg_ridge = LogisticRegression(penalty="l2", solver="saga", random_state=RANDOM_STATE)
logreg_elasticnet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, random_state=RANDOM_STATE)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RANDOM_STATE)
ab = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
xgb = XGBClassifier(random_state=RANDOM_STATE)


In [14]:
def bootstrap_auc_scores(estimator, X, y, n_bootstraps=1000, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    proba = estimator.predict_proba(X)[:, 1]
    y_arr = np.asarray(y)
    n = len(y_arr)
    scores = []
    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_arr[idx])) < 2:
            continue
        scores.append(roc_auc_score(y_arr[idx], proba[idx]))
    return np.array(scores)

def benchmark_models(pipeline, list_model, x_train, y_train, x_val, y_val, random_state=RANDOM_STATE):
    all_cv_result = []

    for name, model in list_model.items():
        classifier = pipeline.set_params(classifier=model)
        classifier.fit(x_train, y_train)

        train_boot = bootstrap_auc_scores(classifier, x_train, y_train, random_state=random_state)
        val_boot   = bootstrap_auc_scores(classifier, x_val, y_val, random_state=random_state)

        all_cv_result.append({
            "name": name,
            "mean_roc_train_score": train_boot.mean(),
            "mean_roc_validate_score": val_boot.mean(),
            "sd_roc_train_score": train_boot.std(),
            "sd_roc_validate_score": val_boot.std(),
        })

    return pd.DataFrame(all_cv_result).sort_values(
        "mean_roc_validate_score", ascending=False
    ).reset_index(drop=True)

In [15]:
models = {
    "LogisticRegressionBase": logreg_base,
    "LogisticRegressionLasso": logreg_lasso,
    "LogisticRegressionRidge": logreg_ridge,
    "LogisticRegressionElasticNet": logreg_elasticnet,
    "DecisionTree": dt,
    "KNearestNeigbor": knn,
    "RandomForest": rf,
    "AdaBoost": ab,
    "GradientBoost": gb,
    "XGBoost": xgb
}

results = benchmark_models(pipeline, models, x_train, y_train, x_val, y_val)

In [16]:
results

,name,mean_roc_train_score,mean_roc_validate_score,sd_roc_train_score,sd_roc_validate_score
0,XGBoost,0.995443,0.827584,0.000250,0.001855
1,LogisticRegressionBase,0.974232,0.822327,0.000893,0.001809
2,LogisticRegressionRidge,0.967054,0.819057,0.001101,0.001862
3,LogisticRegressionElasticNet,0.967022,0.818861,0.001101,0.001863
4,LogisticRegressionLasso,0.974779,0.815311,0.000878,0.001835
5,GradientBoost,0.978994,0.810068,0.000828,0.001868
6,RandomForest,0.999467,0.806236,0.000066,0.002028
7,AdaBoost,0.999824,0.760836,0.000021,0.002219
8,KNearestNeigbor,0.985804,0.694477,0.000598,0.002254
9,DecisionTree,0.999824,0.693330,0.000021,0.001932
